# PulsePredict – 01 Exploratory Data Analysis

**Purpose:** understand the data and justify every cleaning decision *before* any modelling.
Reusable logic lives in `src/` (this notebook only calls it), so the notebook stays readable.

> The bundled `data/raw/cardio_train.csv` is a **synthetic stand-in** with the same schema as the Kaggle
> *Cardiovascular Disease dataset* (70,000 records). If you replace it with the real file, re-run this notebook and
> `python -m src.train` – everything below is computed, nothing is hard-coded.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH
from src.data import load_raw, clean, LIMITS
from src import eda
pd.set_option("display.max_columns", 50)

## 1. Raw data overview

In [ ]:
raw = load_raw(RAW_DATA_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.info()
print("Missing values:", raw.isna().sum().sum())
print("Duplicates (ignoring id):", raw.duplicated(subset=[c for c in raw.columns if c != "id"]).sum())

In [ ]:
raw.describe().T

## 2. Looking for impossible values

Cleaning should be driven by inspection, not by trimming percentiles blindly. Start with blood pressure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
raw["ap_hi"].plot.box(ax=axes[0], vert=False, title="Systolic (raw) – note the extreme values")
raw["ap_hi"].clip(0, 300).hist(bins=80, ax=axes[1]); axes[1].set_title("Systolic (raw, clipped at 300 for display)")
plt.tight_layout()

In [ ]:
print("Systolic < 70 or > 250:", ((raw.ap_hi < 70) | (raw.ap_hi > 250)).sum())
print("Diastolic < 40 or > 150:", ((raw.ap_lo < 40) | (raw.ap_lo > 150)).sum())
print("Systolic <= diastolic (within plausible range):",
      ((raw.ap_hi <= raw.ap_lo) & raw.ap_hi.between(70, 250) & raw.ap_lo.between(40, 150)).sum())
print("Height outside 120-220 cm:", (~raw.height.between(120, 220)).sum())
print("Weight outside 30-250 kg:", (~raw.weight.between(30, 250)).sum())

**Decision.** Values such as 16020, 11500 or −120 mmHg are not extreme patients – they are typing errors
(extra zeros, minus signs, swapped columns). The limits below are *physiological plausibility* bounds, applied as explicit,
counted rules in `src/data.py::clean`. We deliberately do **not** trim to e.g. the 1st–99th percentile, because that would
discard genuine hypertensive patients – exactly the people the model must learn about.

In [ ]:
LIMITS

## 3. Clean, and account for every removed row

In [ ]:
df, report = clean(raw)
print(f"{report['rows_raw']:,} -> {report['rows_clean']:,} rows ({report['percent_removed']}% removed)")
pd.DataFrame(report["steps"])[["rule", "rows_removed"]]

Feature engineering: **age** converted from days to years; **BMI** = weight (kg) / height (m)².
In the deployed pipeline BMI is recomputed inside a `FunctionTransformer`, so training and serving cannot drift apart.

In [ ]:
df.head()

## 4. Class balance

In [ ]:
print(df["target"].value_counts(normalize=True).round(3))
eda.fig_class_balance(df);

Classes are almost perfectly balanced, so plain accuracy is not misleading here and no resampling is needed.
Even so, model selection still uses ROC-AUC, recall and F1 – in a health setting the *type* of error matters, not just the count.

## 5. Distributions and relationships with the outcome

In [ ]:
eda.fig_age_distribution(df); eda.fig_bp_distribution(df); eda.fig_bmi_distribution(df);

In [ ]:
eda.fig_age_vs_outcome(df); eda.fig_bp_vs_outcome(df); eda.fig_cholesterol_vs_outcome(df); eda.fig_lifestyle(df);

In [ ]:
df.corr(numeric_only=True)["target"].drop("target").sort_values(key=abs, ascending=False).round(3)

## 6. Takeaways that shaped modelling

* Systolic blood pressure, age and cholesterol show the clearest relationship with the outcome.
* Relationships are partly **non-linear** (rates climb steeply above ~140 mmHg), which motivates comparing a linear baseline with tree ensembles.
* Classes are balanced; no resampling required.
* Next: `python -m src.train` fits three models on a stratified train split and evaluates on a held-out test set.

*Correlation is not causation, and associations in a training dataset are not individual medical risks.*